In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 20 — Reference coding solutions

Three exercises. Read the lesson and workbook first. Learner functions intentionally raise `NotImplementedError` until you write them. Reference execution is not your assessment.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/shape_lab").is_dir())
if str(ROOT/"src") not in sys.path:sys.path.insert(0,str(ROOT/"src"))


## 20.C1 — `sha256_file`

Compute SHA-256 incrementally from an existing file.

**Before coding:** work a small example on paper.

**Hint:** Read byte chunks; do not hash a filename string in place of its contents.

In [3]:
def sha256_file(path):
    import hashlib
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

In [4]:
import hashlib
path=ROOT/'data/stereo/calibration.json'
assert sha256_file(path)==hashlib.sha256(path.read_bytes()).hexdigest()
assert len(sha256_file(path))==64
print("20.C1: checks passed")

20.C1: checks passed


**Why this works and where it stops:** A hash identifies bytes and detects change. It does not prove a file is accurate, licensed or benign.

## 20.C2 — `validate_geometry_rows`

Validate a finite nonempty Nx3 point table with one unique source ID per row. Return the row count.

**Before coding:** work a small example on paper.

**Hint:** Geometry without its observation identifiers cannot safely be joined back to pixels.

In [5]:
def validate_geometry_rows(points, source_ids):
    p=np.asarray(points,float);ids=list(source_ids)
    if p.ndim!=2 or p.shape[1]!=3 or not len(p) or not np.isfinite(p).all() or len(ids)!=len(p) or len(set(ids))!=len(ids):raise ValueError("Finite XYZ with unique row-aligned IDs required.")
    return len(p)

In [6]:
assert validate_geometry_rows([[1,2,3],[4,5,6]],['frame0:1:2','frame0:1:3'])==2
try:validate_geometry_rows([[1,2,3],[4,5,6]],['same','same'])
except ValueError:pass
else:raise AssertionError("Duplicate observation IDs must not pass.")
print("20.C2: checks passed")

20.C2: checks passed


**Why this works and where it stops:** IDs establish row alignment. Coordinate frame, units, timestamp, preprocessing and source revision are additional required metadata, not implied by this validator.

## 20.C3 — `evidence_status`

Implement a deliberately narrow reporting rule: not_executed if no run, exploratory if no independent holdout, otherwise pass/fail by a finite nonnegative error and predeclared threshold.

**Before coding:** work a small example on paper.

**Hint:** Do not let a low error from a non-run or training example imply acceptance.

In [7]:
def evidence_status(model_run, independent_holdout, error, threshold):
    if not model_run:return 'not_executed'
    if not independent_holdout:return 'exploratory'
    if error is None or not np.isfinite([error,threshold]).all() or error<0 or threshold<0:raise ValueError("Finite nonnegative metric and threshold required.")
    return 'passes_declared_metric' if error<=threshold else 'fails_declared_metric'

In [8]:
assert evidence_status(False,False,None,.01)=='not_executed'
assert evidence_status(True,False,0,.01)=='exploratory'
assert evidence_status(True,True,.02,.01)=='fails_declared_metric'
assert evidence_status(True,True,.005,.01)=='passes_declared_metric'
print("20.C3: checks passed")

20.C3: checks passed


**Why this works and where it stops:** Passing one declared metric is not overall scientific validation or deployment approval. This toy rule teaches the evidence boundary, not an exhaustive acceptance system.